#Imports

In [ ]:
import time
import datetime as dt
from datetime import datetime, timedelta
import numpy as np
import yfinance as yf
import random
import pandas as pd
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning) #ignores "Dataframe is highly fragmented" warning.
from sklearn import preprocessing, svm
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                              StackingClassifier, HistGradientBoostingClassifier,
                              ExtraTreesClassifier, BaggingClassifier, VotingClassifier)
from matplotlib import pyplot as plt
import math
from dateutil.relativedelta import relativedelta
from typing_extensions import reveal_type
from enum import unique

#Fetching Training Data

In [1]:
import pandas as pd

df_earnings_hist_est_eps = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Earnings_History&Estimates&EPS_trend_Balanced(438,61).csv")
print("df_earnings_hist_est_eps shape:", df_earnings_hist_est_eps.shape)
y_earnings_hist_est_eps = df_earnings_hist_est_eps.pop("label")

df_earnings_hist = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Earnings_History_Balanced(505,17).csv")
print("df_earnings_hist shape:", df_earnings_hist.shape)
y_earnings_hist = df_earnings_hist.pop("label")

df_earnings_hist_est = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Earnings_Estimates_&_History_Balanced(487,41).csv")
print("df_earnings_hist_est shape:", df_earnings_hist_est.shape)
y_earnings_hist_est = df_earnings_hist_est.pop("label")

df_earnings_est = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Earnings_Estimates_Balanced(546,16).csv")
print("df_earnings_est shape:", df_earnings_est.shape)
y_earnings_est = df_earnings_est.pop("label")

df_analystpricetarget_eps = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Analyst_price_targets_&_eps_trend_Balanced(517,22).csv")
print("df_analystpricetarget_eps shape:", df_analystpricetarget_eps.shape)
y_analystpricetarget_eps = df_analystpricetarget_eps.pop("label")

df_income_statement = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/IncomeStatement_Balanced_(1608,31).csv")
print("df_income_statement shape:", df_income_statement.shape)
y_income_statement = df_income_statement.pop("label")

df_balancesheet = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Quarterly_Balancesheet_Balanced(792,34).csv")
print("df_balancesheet shape:", df_balancesheet.shape)
y_balancesheet = df_balancesheet.pop("label")

df_cashflow = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Quarterly_Cashflow_Balanced_(786,6).csv")
print("df_cashflow shape:", df_cashflow.shape)
y_cashflow = df_cashflow.pop("label")

df_rev_est = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Revenue_Estimate_Balanced(367,17).csv")
print("df_rev_est shape:", df_rev_est.shape)
y_rev_est = df_rev_est.pop("label")

df_meta = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Meta Data/MetaData[x3]_balanced(2508, 96).csv")
print("df_meta shape:", df_meta.shape)
y_meta = df_meta.pop("label")

df_final_estimator = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Used for testing/FINAL-ESTIMATOR-TRAINING-DATA-BALANCED-.csv")
df_final_estimator = df_final_estimator.drop(columns = ["Unnamed: 0"])
print("df_final_estimator shape: ", df_final_estimator.shape)
y_final_estimator = df_final_estimator.pop("label")

df_earnings_hist_est_eps shape: (438, 61)
df_earnings_hist shape: (504, 17)
df_earnings_hist_est shape: (486, 41)
df_earnings_est shape: (544, 17)
df_analystpricetarget_eps shape: (516, 22)
df_income_statement shape: (1608, 31)
df_balancesheet shape: (792, 35)
df_cashflow shape: (786, 6)
df_rev_est shape: (366, 17)
df_meta shape: (2508, 96)
df_final_estimator shape:  (1124, 11)


In [2]:
print(f'''
df_earnings_hist_est_eps columns: {df_earnings_hist_est_eps.columns}
df_earnings_hist columns: {df_earnings_hist.columns}
df_earnings_hist_est columns: {df_earnings_hist_est.columns}
df_earnings_est columns: {df_earnings_est.columns}
df_analystpricetarget_eps columns: {df_analystpricetarget_eps.columns}
df_income_statement columns: {df_income_statement.columns}
df_balancesheet columns: {df_balancesheet.columns}
df_cashflow columns: {df_cashflow.columns}
df_rev_est columns: {df_rev_est.columns}
df_meta columns: {df_meta.columns}
df_final_estimator columns: {df_final_estimator.columns}
''')


df_earnings_hist_est_eps columns: Index(['epsActual_1', 'epsEstimate_1', 'epsDifference_1', 'surprisePercent_1',
       'epsActual_2', 'epsEstimate_2', 'epsDifference_2', 'surprisePercent_2',
       'epsActual_3', 'epsEstimate_3', 'epsDifference_3', 'surprisePercent_3',
       'epsActual_4', 'epsEstimate_4', 'epsDifference_4', 'surprisePercent_4',
       'avg_1', 'low_1', 'high_1', 'yearAgoEps_1', 'numberOfAnalysts_1',
       'growth_1', 'avg_2', 'low_2', 'high_2', 'yearAgoEps_2',
       'numberOfAnalysts_2', 'growth_2', 'avg_3', 'low_3', 'high_3',
       'yearAgoEps_3', 'numberOfAnalysts_3', 'growth_3', 'avg_4', 'low_4',
       'high_4', 'yearAgoEps_4', 'numberOfAnalysts_4', 'growth_4', 'current_1',
       '7daysAgo_1', '30daysAgo_1', '60daysAgo_1', '90daysAgo_1', 'current_2',
       '7daysAgo_2', '30daysAgo_2', '60daysAgo_2', '90daysAgo_2', 'current_3',
       '7daysAgo_3', '30daysAgo_3', '60daysAgo_3', '90daysAgo_3', 'current_4',
       '7daysAgo_4', '30daysAgo_4', '60daysAgo_4', '

# Algorithms

In [ ]:
GB = HistGradientBoostingClassifier(random_state=1)
BagGB = BaggingClassifier(estimator = GB,  n_estimators = 100, random_state=1)

RF = RandomForestClassifier(random_state=1)

RF2 = RandomForestClassifier(random_state=1)

ADARF = AdaBoostClassifier(estimator = RandomForestClassifier(random_state =1), random_state=1)

ADARF2 = AdaBoostClassifier(estimator = RandomForestClassifier(random_state =1), random_state=1)

ET = ExtraTreesClassifier(random_state=1)

NN = MLPClassifier(solver='adam', alpha=0.1,
                    hidden_layer_sizes=(11, 5),activation = "tanh", random_state=1, max_iter=1000000,  early_stopping=True)
BagNN = BaggingClassifier(estimator = NN,  n_estimators = 10, random_state=1)

vote = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)
vote2 = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)
vote3 = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)
vote4 = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)
vote5 = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)
vote6 = VotingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], voting='soft'
)

stack = StackingClassifier(
    estimators = [('gb', GB), ('rf', RF), ('adarf', ADARF), ('et',ET)], final_estimator=RF
)

QDA = QuadraticDiscriminantAnalysis()

SVM = svm.SVC(random_state=1)

↓ Finds the best random state for the training test split

In [ ]:
# Earnings hist, est, eps trend
clf_earnings_hist_est_eps = RF.fit(df_earnings_hist_est_eps, y_earnings_hist_est_eps)


# Earnings hist, est
clf_earnings_hist_est = vote.fit(df_earnings_hist_est, y_earnings_hist_est)


# Revenue Estimates
clf_rev_est = vote2.fit(df_rev_est, y_rev_est)


# Income Statements
clf_income_statement = vote3.fit(df_income_statement, y_income_statement)


# Balance sheet
clf_balancesheet = vote4.fit(df_balancesheet, y_balancesheet)


# Cashflow
clf_cashflow = stack.fit(df_cashflow, y_cashflow)


# Earnings history
clf_earnings_hist = GB.fit(df_earnings_hist, y_earnings_hist)


# Earnings estimates
clf_earnings_est = ADARF.fit(df_earnings_est, y_earnings_est)


# Analyst Price Targets, eps trend
clf_analystpricetarget_eps = BagGB.fit(df_analystpricetarget_eps, y_analystpricetarget_eps)


# Meta AROUND 58% ACCURACY
clf_meta = ADARF2.fit(df_meta, y_meta)

#Final estimator
clf_final_estimator = vote6.fit(df_final_estimator, y_final_estimator)

Algorithms for each type of fundemental data

In [ ]:
'''
#Earnings hist, est, eps trend
X_train, X_test, y_train, y_test = train_test_split( df_earnings_hist_est_eps, y_earnings_hist_est_eps, random_state=38, test_size=0.05)
clf_earnings_hist_est_eps = RandomForestClassifier(random_state=1).fit(X_train, y_train)
score_earnings_hist_est_eps = clf_earnings_hist_est_eps.score(X_test, y_test)
print("earnings hist, est, and eps trend: ", score_earnings_hist_est_eps)

#Earnings hist, est
X_train, X_test, y_train, y_test = train_test_split( df_earnings_hist_est  , y_earnings_hist_est, random_state=25, test_size=0.05)
clf_earnings_hist_est = vote
clf_earnings_hist_est.fit(X_train, y_train)
score_earnings_hist_est = clf_earnings_hist_est.score(X_test, y_test)
print("earnings hist and est: ", score_earnings_hist_est)

#Earnings history
X_train, X_test, y_train, y_test = train_test_split( df_earnings_hist  , y_earnings_hist, random_state=72, test_size=0.05)
clf_earnings_hist = GB
clf_earnings_hist.fit(X_train, y_train)
score_earnings_hist = clf_earnings_hist.score(X_test, y_test)
print("earnings history: ", score_earnings_hist)

#Earnings estimates
X_train, X_test, y_train, y_test = train_test_split( df_earnings_est  , y_earnings_est, random_state=28, test_size=0.05)
clf_earnings_est = ET
clf_earnings_est.fit(X_train, y_train)
score_earnings_est = clf_earnings_est.score(X_test, y_test)
print("earnings estimates: ", score_earnings_est)

#Analyst Price Targets, eps trend
X_train, X_test, y_train, y_test = train_test_split( df_analystpricetarget_eps  , y_analystpricetarget_eps, random_state=46, test_size=0.05)
clf_analystpricetarget_eps = BagGB
clf_analystpricetarget_eps.fit(X_train, y_train)
score_analystpricetarget_eps = clf_analystpricetarget_eps.score(X_test, y_test)
print("analyst price targets and eps trend: ",score_analystpricetarget_eps)

#Revenue Estimates
X_train, X_test, y_train, y_test = train_test_split( df_rev_est  , y_rev_est, random_state=67, test_size=0.05)
clf_rev_est = vote
clf_rev_est.fit(X_train, y_train)
score_rev_est = clf_rev_est.score(X_test, y_test)
print("revenue estimates: ", score_rev_est)

#Income Statements
X_train, X_test, y_train, y_test = train_test_split( df_income_statement  , y_income_statement, random_state=5, test_size=0.05)
clf_income_statement = vote
clf_income_statement.fit(X_train, y_train)
score_income_statement = clf_income_statement.score(X_test, y_test)
print("income statement: ", score_income_statement)

#Balance sheet
X_train, X_test, y_train, y_test = train_test_split( df_balancesheet  , y_balancesheet, random_state=25, test_size=0.05)
clf_balancesheet = vote
clf_balancesheet.fit(X_train, y_train)
score_balancesheet = clf_balancesheet.score(X_test, y_test)
print("balancesheet: ",score_balancesheet)

#Cashflow
X_train, X_test, y_train, y_test = train_test_split( df_cashflow  , y_cashflow, random_state=21, test_size=0.05)
clf_cashflow = stack
clf_cashflow.fit(X_train, y_train)
score_cashflow = clf_cashflow.score(X_test, y_test)
print("cashflow: ", score_cashflow)

#Meta data is repeated :(
X_train, X_test, y_train, y_test = train_test_split( df_meta  , y_meta, random_state=55, test_size=0.05)
clf_meta = ADARF
clf_meta.fit(X_train, y_train)
score_meta = clf_meta.score(X_test, y_test)
print("meta: ",score_meta)
'''

'\n#Earnings hist, est, eps trend\nX_train, X_test, y_train, y_test = train_test_split( df_earnings_hist_est_eps, y_earnings_hist_est_eps, random_state=38, test_size=0.05)\nclf_earnings_hist_est_eps = RandomForestClassifier(random_state=1).fit(X_train, y_train)\nscore_earnings_hist_est_eps = clf_earnings_hist_est_eps.score(X_test, y_test)\nprint("earnings hist, est, and eps trend: ", score_earnings_hist_est_eps)\n\n#Earnings hist, est\nX_train, X_test, y_train, y_test = train_test_split( df_earnings_hist_est  , y_earnings_hist_est, random_state=25, test_size=0.05)\nclf_earnings_hist_est = vote\nclf_earnings_hist_est.fit(X_train, y_train)\nscore_earnings_hist_est = clf_earnings_hist_est.score(X_test, y_test)\nprint("earnings hist and est: ", score_earnings_hist_est)\n\n#Earnings history\nX_train, X_test, y_train, y_test = train_test_split( df_earnings_hist  , y_earnings_hist, random_state=72, test_size=0.05)\nclf_earnings_hist = GB\nclf_earnings_hist.fit(X_train, y_train)\nscore_earning

In [ ]:
'''
#Cross validation
df = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Used for testing/FINAL-ESTIMATOR-TRAINING-DATA-2_deduped.csv") #abcdefghijklmnopqrstuvwxyz
y = df.pop("label")
print(df.shape, y.shape)

k = int(input("Enter number of cross validations: "))

RFscore = cross_val_score(RF,df,y,cv=k)
GBscore = cross_val_score(GB,df,y,cv=k)
BagGBscore = cross_val_score(BagGB,df,y,cv=k)
ADARFscore = cross_val_score(ADARF,df,y,cv=k)
ETscore = cross_val_score(ET,df,y,cv=k)
#BagNNscore = cross_val_score(BagNN,df,y,cv=k)
Votescore = cross_val_score(vote,df,y,cv=k)
Stackscore = cross_val_score(stack,df,y,cv=k)
'''
'''
print(f

{k} cross validations

RFscore: {RFscore.mean():.3f} (SD: {RFscore.std():.3f})
{RFscore}

GBscore: {GBscore.mean():.3f} (SD: {GBscore.std():.3f})
{GBscore}

BagGBscore: {BagGBscore.mean():.3f} (SD: {BagGBscore.std():.3f})
{BagGBscore}

ADARFscore: {ADARFscore.mean():.3f} (SD: {ADARFscore.std():.3f})
{ADARFscore}

ETscore: {ETscore.mean():.3f} (SD: {ETscore.std():.3f})
{ETscore}


Votescore: {Votescore.mean():.3f} (SD: {Votescore.std():.3f})
{Votescore}

Stackscore: {Stackscore.mean():.3f} (SD: {Stackscore.std():.3f})
{Stackscore}
      )
'''


'\nprint(f\n\n{k} cross validations\n\nRFscore: {RFscore.mean():.3f} (SD: {RFscore.std():.3f})\n{RFscore}\n\nGBscore: {GBscore.mean():.3f} (SD: {GBscore.std():.3f})\n{GBscore}\n\nBagGBscore: {BagGBscore.mean():.3f} (SD: {BagGBscore.std():.3f})\n{BagGBscore}\n\nADARFscore: {ADARFscore.mean():.3f} (SD: {ADARFscore.std():.3f})\n{ADARFscore}\n\nETscore: {ETscore.mean():.3f} (SD: {ETscore.std():.3f})\n{ETscore}\n\n\nVotescore: {Votescore.mean():.3f} (SD: {Votescore.std():.3f})\n{Votescore}\n\nStackscore: {Stackscore.mean():.3f} (SD: {Stackscore.std():.3f})\n{Stackscore}\n      )\n'

#Functions (fetching testing data, building the model, and est_date function)

In [ ]:
def unique_names(df, repeats):
  shape = df.shape
  columns_no = shape[1]/repeats
  columns_names = []
  for i in range(int(columns_no)):
    columns_names.append(df.columns[i])

  count = 1

  new_names = []

  for i in range(repeats):
    for i in range(int(columns_no)):
      if df.columns[i] in columns_names:
        new_column = f"{df.columns[i]}_{count}"
        new_names.append(new_column)

      else:
        new_column = df.columns[i]
        new_names.append(new_column)
    count = count+1
  return new_names


def testing_data(stock):

  stock = yf.Ticker(stock)

  try:
    #balancesheet
    balancesheet = stock.quarterly_balancesheet
    balancesheet = pd.concat([balancesheet.iloc[:,0], balancesheet.iloc[:,1], balancesheet.iloc[:,2],balancesheet.iloc[:,3],balancesheet.iloc[:,4]])
    balancesheet = balancesheet.to_frame().T
    balancesheet.columns = unique_names(balancesheet, 5)
    #balancesheet = balancesheet.dropna(axis=1)
  except:
    balancesheet = pd.DataFrame()
    print(stock, "balancesheet failed to load")

  try:
    #cashflow
    cashflow = stock.quarterly_cashflow
    cashflow = pd.concat([cashflow.iloc[:,0], cashflow.iloc[:,1], cashflow.iloc[:,2],cashflow.iloc[:,3],cashflow.iloc[:,4]])
    cashflow = cashflow.to_frame().T
    cashflow.columns = unique_names(cashflow, 5)
    #cashflow = cashflow.dropna(axis=1)
  except:
    cashflow = pd.DataFrame()
    print(str(stock),"cashflow failed to load")

  try:
    #Analyst price targets & eps trend
    analyst_price_targets = pd.DataFrame.from_dict(stock.analyst_price_targets, orient='index').transpose()
    analyst_price_targets.rename(columns = {"current":"current price"}, inplace = True)
    analyst_price_targets.rename(columns = {"high":"high estimate"}, inplace = True)
    analyst_price_targets.rename(columns = {"low":"low estimate"}, inplace = True)
    analyst_price_targets.rename(columns = {"mean":"mean estimate"}, inplace = True)
    analyst_price_targets.rename(columns = {"median":"median estimate"}, inplace = True)
    #analyst_price_targets = analyst_price_targets.dropna(axis=1)
  except:
    analyst_price_targets = pd.DataFrame()
    print(stock,"analyst price targets failed to load")

  try:
    eps_trend = stock.eps_trend
    eps_trend = pd.concat([eps_trend.iloc[0,:],eps_trend.iloc[1,:],eps_trend.iloc[2,:],eps_trend.iloc[3,:]]).to_frame().T
    eps_trend.columns = unique_names(eps_trend, 4)
    #eps_trend = eps_trend.dropna(axis=1)
  except:
    eps_trend = pd.DataFrame()
    print(stock,"eps trend failed to load")


  analyst_price_targets_eps_trend = pd.concat([analyst_price_targets, eps_trend], axis=1)

  try:
    #income statement
    incomestmt = stock.income_stmt
    incomestmt = pd.concat([incomestmt.iloc[:,0], incomestmt.iloc[:,1], incomestmt.iloc[:,2],incomestmt.iloc[:,3],incomestmt.iloc[:,4]])
    incomestmt = incomestmt.to_frame().T
    incomestmt.columns = unique_names(incomestmt,5)
    #incomestmt = incomestmt.dropna(axis=1)
  except:
    incomestmt = pd.DataFrame()
    print(stock,"incomestmt failed to load")

  try:
    #earning estimate
    earning_est = stock.earnings_estimate
    earning_est = pd.concat([earning_est.iloc[0,:],earning_est.iloc[1,:],earning_est.iloc[2,:],earning_est.iloc[3,:]]).to_frame().T
    earning_est.columns = unique_names(earning_est, 4)
    #earning_est = earning_est.dropna(axis=1)
  except:
    earning_est = pd.DataFrame()
    print(stock,"earnings estimates failed to load")

  try:
    #earnings history
    earning_hist = stock.earnings_history
    earning_hist = pd.concat([earning_hist.iloc[0,:],earning_hist.iloc[1,:],earning_hist.iloc[2,:],earning_hist.iloc[3,:]]).to_frame().T
    earning_hist.columns = unique_names(earning_hist, 4)
    #earnins_hist = earning_hist.dropna(axis=1)
  except:
    earning_hist = pd.DataFrame()
    print(stock,"earnings history failed to load")

  #earnings hist + est
  earning_hist = earning_hist.reset_index(drop=True)
  earning_est = earning_est.reset_index(drop=True)
  eps_trend = eps_trend.reset_index(drop=True)
  earnings_hist_est = pd.concat([earning_hist, earning_est], axis=1)
  #earnings_hist_est = earnings_hist_est.dropna()

  #earnings hist + estimates + eps
  earnings_hist_est_eps = pd.concat([earning_hist, earning_est, eps_trend], axis=1)
  #earnings_hist_est_eps = earnings_hist_est_eps.dropna()

  try:
    #Revenue estimates
    rev_est = stock.revenue_estimate
    rev_est = pd.concat([rev_est.iloc[0,:],rev_est.iloc[1,:],rev_est.iloc[2,:],rev_est.iloc[3,:]]).to_frame().T
    rev_est.columns = unique_names(rev_est, 4)
    #rev_est = rev_est.dropna(axis=1)
  except:
    rev_est = pd.DataFrame()
    print(stock,"revenue estimates failed to load")

  try:
    #Meta data
    rev_and_earning_est = pd.concat([rev_est,earning_est], axis = 1)
    rev_and_earning_est.columns = unique_names(rev_and_earning_est,2)
    meta = pd.concat([balancesheet, cashflow, analyst_price_targets, eps_trend, rev_and_earning_est, earning_hist  ],axis=1)
  except:
    meta = pd.DataFrame
    print(stock,"meta failed to load")

  return balancesheet, cashflow, analyst_price_targets_eps_trend, incomestmt, earning_est, earning_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta


def transform(input, template):

  df = template.iloc[0].to_frame().T

  input = pd.concat([input, df])

  input = input.dropna(axis=1)

  if input.shape[0] !=1:
    input = input.iloc[0].to_frame().T
  elif input.shape[0] == 1:
    input = pd.DataFrame()

  return input


def transformed_testing_data(stock):

  failed = []
  succeeded = []


  balancesheet, cashflow, analyst_target_eps, incomestmt, earnings_est, earnings_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta = testing_data(stock)

  balancesheet = transform(balancesheet, df_balancesheet)
  cashflow = transform(cashflow, df_cashflow)
  analyst_target_eps = transform(analyst_target_eps, df_analystpricetarget_eps)
  incomestmt = transform(incomestmt, df_income_statement)
  earnings_est = transform(earnings_est, df_earnings_est)
  earnings_hist = transform(earnings_hist, df_earnings_hist)
  earnings_hist_est = transform(earnings_hist_est, df_earnings_hist_est)
  earnings_hist_est_eps = transform(earnings_hist_est_eps, df_earnings_hist_est_eps)
  rev_est = transform(rev_est, df_rev_est)
  meta = transform(meta, df_meta)

  if transform(balancesheet, df_balancesheet).shape != (1, 34):
      failed.append(("balancesheet"))
  else:
      succeeded.append(("balancesheet"))

  if cashflow.shape != (1, 5):
      failed.append(( "cashflow"))
  else:
      succeeded.append(( "cashflow"))

  if analyst_target_eps.shape != (1, 21):
      failed.append(( "analyst_target_eps"))
  else:
      succeeded.append(( "analyst_target_eps"))

  if incomestmt.shape != (1, 30):
      failed.append(( "incomestmt"))
  else:
      succeeded.append(("incomestmt"))

  if earnings_est.shape != (1, 16):
      failed.append(( "earnings_est"))
  else:
      succeeded.append(( "earnings_est"))

  if earnings_hist.shape != (1, 16):
      failed.append(( "earnings_hist"))
  else:
      succeeded.append(( "earnings_hist"))

  if earnings_hist_est.shape != (1, 40):
      failed.append(( "earnings_hist_est"))
  else:
      succeeded.append(( "earnings_hist_est"))

  if earnings_hist_est_eps.shape != (1, 60):
      failed.append(( "earnings_hist_est_eps"))
  else:
      succeeded.append(("earnings_hist_est_eps"))

  if rev_est.shape != (1, 16):
      failed.append(("rev_est"))
  else:
      succeeded.append(("rev_est"))

  if meta.shape != (1, 95):
      failed.append(( "meta"))
  else:
      succeeded.append(( "meta"))

  return succeeded, failed, balancesheet, cashflow, analyst_target_eps, incomestmt, earnings_est, earnings_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta


def fundemental_model(stock):


  weighted_prediction = []
  predictions = []

  successful, failed, balancesheet, cashflow, analyst_target_eps, incomestmt, earnings_est, earnings_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta = transformed_testing_data(stock)
  print(f'''
  {stock}
  Successful: {len(successful)}
        {successful}
  Failed: {len(failed)}
        {failed}
  ''')
  if len(failed)==10:
    print("🔺🔺🔺 error occured 🔺🔺🔺")
    return f"error with {stock}",f"error with {stock}",f"error with {stock}",f"error with {stock}"

  else:
    #if "balancesheet" in successful:
    try:
        pred = clf_balancesheet.predict_proba(balancesheet)
        #weighted_prediction.append(pred[0,1]*(3025/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3025/32906))
        predictions.append(-100)
        print(f"Error occurred with balancesheet: {e}")

    #if "cashflow" in successful:
    try:
        pred = clf_cashflow.predict_proba(cashflow)
        #weighted_prediction.append(pred[0,1]*(3025/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3025/32906))
        predictions.append(-100)
        print(f"Error occurred with cashflow: {e}")

    #if "analyst_target_eps" in successful:
    try:
        pred = clf_analystpricetarget_eps.predict_proba(analyst_target_eps)
        #weighted_prediction.append(pred[0,1]*(3025/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3025/32906))
        predictions.append(-100)
        print(f"Error occurred with analyst_target_eps: {e}")

    #if "incomestmt" in successful:
    try:
        pred = clf_income_statement.predict_proba(incomestmt)
        #weighted_prediction.append(pred[0,1]*(3481/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3481/32906))
        predictions.append(-100)
        print(f"Error occurred with incomestmt: {e}")

    #if "earnings_est" in successful:
    try:
        pred = clf_earnings_est.predict_proba(earnings_est)
        #weighted_prediction.append(pred[0,1]*(2916/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(2916/32906))
        predictions.append(-100)
        print(f"Error occurred with earnings_est: {e}")

    #if "earnings_hist" in successful:
    try:
        pred = clf_earnings_hist.predict_proba(earnings_hist)
        #weighted_prediction.append(pred[0,1]*(3600/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3600/32906))
        predictions.append(-100)
        print(f"Error occurred with earnings_hist: {e}")

    #if "earnings_hist_est" in successful:
    try:
        pred = clf_earnings_hist_est.predict_proba(earnings_hist_est)
        #weighted_prediction.append(pred[0,1]*(4225/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(4225/32906))
        predictions.append(-100)
        print(f"Error occurred with earnings_hist_est: {e}")

    #if "earnings_hist_est_eps" in successful:
    try:
        pred = clf_earnings_hist_est_eps.predict_proba(earnings_hist_est_eps)
        #weighted_prediction.append(pred[0,1]*(3844/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3844/32906))
        predictions.append(-100)
        print(f"Error occurred with earnings_hist_est_eps: {e}")

    #if "rev_est" in successful:
    try:
        pred = clf_rev_est.predict_proba(rev_est)
        #weighted_prediction.append(pred[0,1]*(2401/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(2401/32906))
        predictions.append(-100)
        print(f"Error occurred with rev_est: {e}")

    #if "meta" in successful:
    try:
        pred = clf_meta.predict_proba(meta)
        #weighted_prediction.append(pred[0,1]*(3364/32906))
        predictions.append(pred[0,1])
    except Exception as e:
        #weighted_prediction.append(0.5*(3364/32906))
        predictions.append(-100)
        print(f"Error occurred with meta: {e}")


    predictions = pd.DataFrame({"":predictions}).transpose()
    final_estimator_pred = clf_final_estimator.predict_proba(predictions)
    if final_estimator_pred[0,1] >0.91:
      prediction = 1
      print("🔹", prediction,"🔹", "~83% accuracy 🔹")

    elif final_estimator_pred[0,1]<0.1:
      prediction = -1
      print("🔹", prediction,"🔹", "~86% accuracy 🔹")

    else:
       if final_estimator_pred[0,1] >0.8:
        prediction = 1
        print("🔹", prediction,"🔹", "~69% accuracy 🔹")

       elif final_estimator_pred[0,1]<0.15:
        prediction = -1
        print("🔹", prediction,"🔹", "~69% accuracy 🔹")
       else:
        prediction = "unknown"
        print("🔸 Uncertain 🔸")


    return prediction, final_estimator_pred[0,1]

In [ ]:
def change_days(date_str, days):
    import datetime as dt
    date = dt.datetime.strptime(date_str, "%Y-%m-%d")

    new_date = date + timedelta(days=days)

    return new_date.strftime("%Y-%m-%d")

def change_months(date_str, monthss):
  import datetime as dt
  date_obj = datetime.strptime(date_str, "%Y-%m-%d")

  new_date = date_obj + relativedelta(months=monthss)

  return new_date.strftime("%Y-%m-%d")

def est_date(stock):
  ticker = yf.Ticker(stock)
  ticker = ticker.quarterly_balance_sheet
  date = ((str(ticker.columns[0])).split())[0]
  date = change_months(date, 3)
  return date

#Tests


In [ ]:
#UNSEEN: FOR TESTING FINAL ESTIMATOR
stocks=['LOVE', 'LOWLF', 'LOW', 'LPLA', 'LQWDF', 'LXU', 'LYTS', 'LTC', 'LUCMF', 'LUCRF', 'LGCL', 'LUCN', 'LUCD', 'LCID', 'LKNCY', 'LUCK', 'LDSN', 'LUDG', 'LU', 'LKFLF', 'LULU', 'LVLU', 'LUMB', 'LUMN', 'LFT', 'LITE', 'LMGDF', 'LRGR', 'LAZR', 'LMGIF', 'LDXHF', 'LUGDF', 'LUNMF', 'LPKGF', 'LUVU', 'LUXE', 'LXFR', 'LUXHP', 'LUXFF', 'LVMUY', 'LVMHF', 'LXP', 'YAHOF', 'YAHOY', 'LCXEF', 'LYEL', 'LYFT', 'LYSCF', 'LYSDY', 'LYB', 'SLMNP', 'LYBC', 'LYRA', 'LYTHF', 'MFBP', 'MGPUF', 'MTB', 'MHO', 'MTWO', 'MTHRF', 'MTHRY', 'MLGCF', 'MBAVU', 'MBAV', 'MAAS', 'MTAL', 'MAC', 'MNR', 'MACT', 'MKZR', 'MTSI', 'MQBKY', 'DBMBF', 'MRPT', 'MGNX', 'M', 'MADGF', 'MCBK', 'MSET', 'MSGE', 'MSGS', 'MDGL', 'MMCP', 'MAG', 'MGLUY', 'MALJF', 'MAGE', 'MEGL', 'MGA', 'MGMNF', 'BRIOF', 'MX', 'MAGN', 'MNSEF', 'MGY', 'MYTAY', 'MAHMF', 'MAIA', 'MAIN', 'MSWV', 'MNSBP', 'MEQYF', 'MSCH', 'MYNZ', 'MASN', 'MSS', 'MJGCF', 'MJDLF', 'MMYT', 'MKTAY', 'MAKOF', 'MLGF', 'MLYBY', 'MBUU', 'MAMA', 'TUSK', 'MNGPF', 'MAWHF', 'MANU', 'MNDJF', 'MAORF', 'MANDF','MNXXF', 'MGRX', 'MANH', 'MTW', 'MANVF', 'MNKD', 'MAN', 'MFC', 'MPFRF', 'MPFRY', 'MGMLF', 'MLFNF', 'MGWFF', 'CART', 'MAPGF', 'MAQC', 'MARA', 'MBBC', 'MPC', 'MRVI', 'MRPMF', 'MMI', 'MCS', 'MRX', 'MRRTY', 'MAJI', 'MARIF', 'MRMD', 'MRIN', 'MBOF', 'MARPS', 'HZO', 'MTEK', 'MAXQF', 'MRTMD', 'MKL', 'MRKR', 'MKTX', 'MAAL', 'MKTW', 'RVBR', 'MAKSY', 'MRLWF', 'MQ', 'MNAT', 'TMGID', 'MAR', 'VAC', 'MMC', 'MRTN', 'MRT', 'MLM', 'MRETF', 'MARUF', 'MARUY', 'MAURY', 'MBCOF', 'MARVF', 'MRVL', 'MVNC', 'MGLD', 'MAS', 'MASI', 'GNYPF', 'MGPHF', 'MMMW', 'MSSEL', 'MAMO', 'MTZ', 'MHH', 'MBC', 'MA', 'MCFT', 'MMND', 'MTDR', 'MTCH', 'MTLS', 'MTRN', 'MTRN', 'MTRX', 'MATX', 'MAT', 'MATW', 'MTTRF', 'MCHT', 'MKGP', 'MIGI', 'MAXXF', 'MXCT', 'MMS', 'MXL', 'MRTI', 'MAYX', 'MFGCF', 'MAYNF', 'MEC', 'MZDAF', 'MZDAY', 'MBI', 'MBKL', 'MBX', 'MSMY', 'MAMTF', 'MCBRF', 'MKC', 'MCCRF', 'MCDIF', 'MCD', 'MDNDF', 'MUX', 'MCFNF', 'MGRC', 'GLFN', 'MCK', 'MCRAA', 'MCRAB']

len(stocks)

238

In [ ]:
print(len(error), len(stocks) - len(error))
for i in range(len(error)):
  stocks.remove(error[i])

print(len(stocks))
print(stocks)

0 238
238
['LOVE', 'LOWLF', 'LOW', 'LPLA', 'LQWDF', 'LXU', 'LYTS', 'LTC', 'LUCMF', 'LUCRF', 'LGCL', 'LUCN', 'LUCD', 'LCID', 'LKNCY', 'LUCK', 'LDSN', 'LUDG', 'LU', 'LKFLF', 'LULU', 'LVLU', 'LUMB', 'LUMN', 'LFT', 'LITE', 'LMGDF', 'LRGR', 'LAZR', 'LMGIF', 'LDXHF', 'LUGDF', 'LUNMF', 'LPKGF', 'LUVU', 'LUXE', 'LXFR', 'LUXHP', 'LUXFF', 'LVMUY', 'LVMHF', 'LXP', 'YAHOF', 'YAHOY', 'LCXEF', 'LYEL', 'LYFT', 'LYSCF', 'LYSDY', 'LYB', 'SLMNP', 'LYBC', 'LYRA', 'LYTHF', 'MFBP', 'MGPUF', 'MTB', 'MHO', 'MTWO', 'MTHRF', 'MTHRY', 'MLGCF', 'MBAVU', 'MBAV', 'MAAS', 'MTAL', 'MAC', 'MNR', 'MACT', 'MKZR', 'MTSI', 'MQBKY', 'DBMBF', 'MRPT', 'MGNX', 'M', 'MADGF', 'MCBK', 'MSET', 'MSGE', 'MSGS', 'MDGL', 'MMCP', 'MAG', 'MGLUY', 'MALJF', 'MAGE', 'MEGL', 'MGA', 'MGMNF', 'BRIOF', 'MX', 'MAGN', 'MNSEF', 'MGY', 'MYTAY', 'MAHMF', 'MAIA', 'MAIN', 'MSWV', 'MNSBP', 'MEQYF', 'MSCH', 'MYNZ', 'MASN', 'MSS', 'MJGCF', 'MJDLF', 'MMYT', 'MKTAY', 'MAKOF', 'MLGF', 'MLYBY', 'MBUU', 'MAMA', 'TUSK', 'MNGPF', 'MAWHF', 'MANU', 'MNDJF', 'M

In [ ]:
predictions = []
error = []

for i in range(len(stocks)):
  try:
    weighted,final_est,preds = fundemental_model(stocks[i])
    print(i, stocks[i])

    if final_est > 0.9:
      prediction = 1
    elif final_est <0.1:
      prediction = -1
    else:
      prediction = 0

    predictions.append(prediction)
  except:
    error.append(stocks[i])

predictions, error


 LOVE
 Successful: 10
       ['balancesheet', 'cashflow', 'analyst_target_eps', 'incomestmt', 'earnings_est', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'rev_est', 'meta']
 Failed: 0
       []
 


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


🔹 0.4468288549494008 🔹 0.4061109482720391 🔹
0 LOVE
yfinance.Ticker object <LOWLF> balancesheet failed to load
yfinance.Ticker object <LOWLF> cashflow failed to load
yfinance.Ticker object <LOWLF> incomestmt failed to load
yfinance.Ticker object <LOWLF> earnings history failed to load

 LOWLF
 Successful: 3
       ['analyst_target_eps', 'earnings_est', 'rev_est']
 Failed: 7
       ['balancesheet', 'cashflow', 'incomestmt', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'meta']
 
Error occurred with balancesheet: at least one array or dtype is required
Error occurred with cashflow: at least one array or dtype is required


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


Error occurred with incomestmt: at least one array or dtype is required
Error occurred with earnings_hist: at least one array or dtype is required
Error occurred with earnings_hist_est: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with earnings_hist_est_eps: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with meta: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Common Stock Equity_1
- Common Stock Equity_2
- Common Stock Equity_3
- Common Stock Equity_4
- Common Stock Equity_5
- ...

🔹 0.4575668832202 🔹 0.45806042407909303 🔹
1 LOWLF


...

🔹 0.4798729026779899 🔹 0.2310260550903793 🔹
236 MCRAA
yfinance.Ticker object <MCRAB> balancesheet failed to load
yfinance.Ticker object <MCRAB> cashflow failed to load
yfinance.Ticker object <MCRAB> incomestmt failed to load
yfinance.Ticker object <MCRAB> earnings history failed to load

 MCRAB
 Successful: 3
       ['analyst_target_eps', 'earnings_est', 'rev_est']
 Failed: 7
       ['balancesheet', 'cashflow', 'incomestmt', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'meta']
 
Error occurred with balancesheet: at least one array or dtype is required
Error occurred with cashflow: at least one array or dtype is required


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


Error occurred with incomestmt: at least one array or dtype is required
Error occurred with earnings_hist: at least one array or dtype is required
Error occurred with earnings_hist_est: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with earnings_hist_est_eps: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with meta: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Common Stock Equity_1
- Common Stock Equity_2
- Common Stock Equity_3
- Common Stock Equity_4
- Common Stock Equity_5
- ...

🔹 0.48081440835603495 🔹 0.2960819784882573 🔹
237 MCRAB


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


([np.float64(0.4061109482720391),
  np.float64(0.45806042407909303),
  np.float64(0.9332696601713408),
  np.float64(0.9001412527201273),
  np.float64(0.6381591789377269),
  np.float64(0.3209963098378987),
  np.float64(0.16009121168393284),
  np.float64(0.4224334176242472),
  np.float64(0.5825331993969588),
  np.float64(0.7089187542278808),
  np.float64(0.757027039814911),
  np.float64(0.6447308645207822),
  np.float64(0.1569352808018133),
  np.float64(0.6069486410506952),
  np.float64(0.8887499122051088),
  np.float64(0.2346607557052426),
  np.float64(0.9200322267013091),
  np.float64(0.31932737244392206),
  np.float64(0.5791860054117007),
  np.float64(0.5028652946344497),
  np.float64(0.9401386853777152),
  np.float64(0.1763527196587094),
  np.float64(0.24224062279566064),
  np.float64(0.3333558539357718),
  np.float64(0.6300631765848578),
  np.float64(0.6203445067785902),
  np.float64(0.4835743826633062),
  np.float64(0.45806042407909303),
  np.float64(0.20642912834307225),
  np.floa

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Used for testing//Final Estimator Predictions Test-2.csv")
df.shape

(238, 1)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/SP/Financial Data/Used Files/Used for testing//Final Estimator Predictions Test-2.csv")
df["label"] = np.where(df["label"] > 0.91, 1, np.where(df["label"] < 0.1, -1, 0))
print(df.to_string())

     label
0        0
1        0
2        1
3        0
4        0
5        0
6        0
7        0
8        0
9        0
10       0
11       0
12       0
13       0
14       0
15       0
16       1
17       0
18       0
19       0
20       1
21       0
22       0
23       0
24       0
25       0
26       0
27       0
28       0
29       0
30       0
31       0
32       0
33       0
34       0
35       0
36       0
37       0
38       0
39       0
40       0
41       0
42       0
43       0
44       0
45       0
46       0
47       0
48       0
49       0
50       0
51       0
52       0
53       0
54       0
55       0
56       1
57       0
58       0
59       0
60       0
61       0
62       0
63       0
64       0
65       0
66       0
67       0
68       0
69       0
70       0
71       0
72       0
73       0
74       0
75       0
76       0
77       0
78       0
79       0
80       0
81       0
82       0
83       0
84       1
85       0
86       0
87       0
88       0
89       0

In [ ]:
y = pd.DataFrame({"":label})

cr = classification_report(y[""],df)
print(cr)
cm = confusion_matrix(y[""],df)
print(cm)

              precision    recall  f1-score   support

          -1       0.86      0.06      0.11       106
           0       0.00      0.00      0.00         0
           1       0.83      0.11      0.20       132

    accuracy                           0.09       238
   macro avg       0.56      0.06      0.10       238
weighted avg       0.84      0.09      0.16       238

[[  6  97   3]
 [  0   0   0]
 [  1 116  15]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
label = []
error = []

for i in range(len(stocks)):
    try:
        data = yf.Ticker(stocks[i]).history(start="2025-06-01", end="2025-06-04")
        recent_close = float(data["Close"].tail(1).iloc[0])

        data = yf.Ticker(stocks[i]).history(start="2025-03-25", end="2025-03-31")
        past_close = float(data["Close"].tail(1).iloc[0])

        change = (recent_close - past_close)/past_close
        print(i, stocks[i], change)

        if change > 0.0000:
            label.append(1)
        elif change == 0.0:
            error.append(stocks[i])
        elif change<-0.0000:
            label.append(-1)
        else:
            label.append(0)
    except Exception as e:
        print(stocks[i], "failed:", e)
        error.append(stocks[i])

print(len(label))

0 LOVE 0.04825735318707736
1 LOWLF 0.07692308794464563
2 LOW 0.006949564002780175
3 LPLA 0.14669185462674197
4 LQWDF 1.394736941171156
5 LXU 0.18664643256350544
6 LYTS -0.05171033025650752
7 LTC 0.0056460123889593
8 LUCMF 0.14609429737206772
9 LUCRF -0.3199999928474426
10 LGCL 0.17021279428921557
11 LUCN 0.16898147918191966
12 LUCD -0.20253167326605756
13 LCID -0.05603443093234513
14 LKNCY -0.041279125600454394
15 LUCK -0.13491020082212826
16 LDSN 0.11111118297242212
17 LUDG -0.3088235906405812
18 LU -0.027027000903735277
19 LKFLF 0.17073178106940157
20 LULU 0.14280354347968705
21 LVLU 0.04370181881299501
22 LUMB 0.16099582416064603
23 LUMN 0.04314722685914751
24 LFT -0.02307699260727251
25 LITE 0.24081174902340582
26 LMGDF 1.1333334657881036
27 LRGR 0.35294116787706953
28 LAZR -0.4289276776349257
29 LMGIF 0.19215079872861893
30 LDXHF 0.33200005190571263
31 LUGDF 0.6642528890713504
32 LUNMF 0.1778846815140314
33 LPKGF 0.6250366863900143
34 LUVU -0.20000002980232195
35 LUXE 0.2771084644

In [ ]:
X = pd.DataFrame({"":predictions})
y = pd.DataFrame({"":label})

cr = classification_report(y[""],X[""])
print(cr)
cm = confusion_matrix(y[""],X[""])
print(cm)

              precision    recall  f1-score   support

          -1       0.57      0.04      0.08        97
           0       0.06      0.67      0.11        18
           1       0.67      0.11      0.19       123

    accuracy                           0.13       238
   macro avg       0.43      0.27      0.13       238
weighted avg       0.58      0.13      0.14       238

[[  4  90   3]
 [  2  12   4]
 [  1 108  14]]


#Other

In [ ]:
df = pd.DataFrame()
error = []

for i in range(len(stocks)):
  try:
    weighted_pred,final_est, predictions = fundemental_model(stocks[i])
    df = pd.concat([df,predictions])
  except:
    print(stocks[i])
    error.append(stocks[i])
print(df)

yfinance.Ticker object <MNXXF> incomestmt failed to load
yfinance.Ticker object <MNXXF> earnings history failed to load

 MNXXF
 Successful: 5
       ['balancesheet', 'cashflow', 'analyst_target_eps', 'earnings_est', 'rev_est']
 Failed: 5
       ['incomestmt', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'meta']
 


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warni

Error occurred with incomestmt: at least one array or dtype is required
Error occurred with earnings_hist: at least one array or dtype is required
Error occurred with earnings_hist_est: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with earnings_hist_est_eps: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with meta: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_4
- epsEstimate_1



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


🔹 0.46106722554013646 🔹 0.9143349674402002 🔹
yfinance.Ticker object <MGRX> incomestmt failed to load
yfinance.Ticker object <MGRX> earnings history failed to load

 MGRX
 Successful: 5
       ['balancesheet', 'cashflow', 'analyst_target_eps', 'earnings_est', 'rev_est']
 Failed: 5
       ['incomestmt', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'meta']
 


...

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(


Error occurred with incomestmt: at least one array or dtype is required
Error occurred with earnings_hist: at least one array or dtype is required
Error occurred with earnings_hist_est: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with earnings_hist_est_eps: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- epsActual_1
- epsActual_2
- epsActual_3
- epsActual_4
- epsDifference_1
- ...

Error occurred with meta: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Common Stock Equity_1
- Common Stock Equity_2
- Common Stock Equity_3
- Common Stock Equity_4
- Common Stock Equity_5
- ...



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(


🔹 0.48081440835603495 🔹 0.3230059901565094 🔹
             0       1         2           3         4           5  \
      0.528077    0.51  0.119555 -100.000000  0.119203 -100.000000   
      0.625699    0.93  0.119555 -100.000000  0.119203 -100.000000   
      0.775064    0.50  0.861756 -100.000000  0.880797    0.268696   
      0.347257    0.55  0.615668    0.687325  0.119203    0.095745   
      0.248270    0.53  0.119555    0.176493  0.119203 -100.000000   
..         ...     ...       ...         ...       ...         ...   
      0.894095    0.31  0.119555    0.150421  0.119203 -100.000000   
      0.607414    0.43  0.092968 -100.000000  0.119203    0.835533   
   -100.000000 -100.00  0.671120    0.447968  0.880797    0.780457   
   -100.000000 -100.00  0.362200 -100.000000  0.119203 -100.000000   
   -100.000000 -100.00  0.372442 -100.000000  0.119203 -100.000000   

             6       7         8           9  
   -100.000000 -100.00  0.860245 -100.000000  
   -100.000000 -100.

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


In [ ]:
for i in range(len(l)):
  x = est_date(l[i])
  print(x)

2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30
2025-09-30

In [ ]:
est_date("aapl")

'2025-09-30'

In [ ]:
df = pd.DataFrame({"":predictions})
df.shape

(238, 1)

In [ ]:
df.to_csv('Final Estimator Predictions Test.csv', index=False)

In [ ]:
X[""].corr(y[""])

np.float64(0.17142024712767748)

In [ ]:
df = pd.DataFrame({"":label})
df.to_csv("Labels for final estimator.csv")

In [ ]:
print((X[""]==1).sum().sum())

9


In [ ]:
K = pd.DataFrame({"pred":predictions, "label":label})
K.to_csv("RESULTS FROM FUNDEMENTAL MODEL(UNCHECKED, SUPER POSITIVE).csv")

In [ ]:
K["pred"].corr(K["label"])

np.float64(0.9444371145462145)

In [ ]:
#THE ORIGINAL ONE DONT EDIT THIS, ALL OF THESE STOCKS HAVE LABELS!
stocks = ['LAZ', 'LZB', 'GORV', 'LCII', 'LCNB', 'LRE', 'LEMIF', 'LBUY', 'LFLY', 'LPTX', 'LCCN', 'LEA', 'LTRE', 'LEAT', 'LDTC', 'LFGP', 'LMPMY', 'LEE', 'LEEEF', 'LPPI', 'LTESF', 'LEGIF', 'LGCY', 'LEGH', 'LGGNF', 'LGGNY', 'LZ', 'LEGT', 'LEGN', 'LPSIF', 'LGBS', 'LEG', 'LEBGF', 'LGRDY', 'LEHKQ', 'LEHLQ', 'LHHMQ', 'LEHNQ', 'LSE', 'LEJUY', 'LNTO', 'LMAT', 'LMND', 'LC', 'TREE', 'LDWY', 'LNVGF', 'LNVGY', 'LNSR', 'LASLY', 'LENZ', 'LNZNF', 'LLLAF', 'LECRF', 'DRS', 'FINMF', 'FINMY', 'LEFUF', 'LEEN', 'LYSFY', 'LSAK', 'LESL', 'AIUG', 'LEVI', 'LVXFF', 'LWCL', 'LEXX', 'LXEO', 'LXRX', 'LX', 'LIFD', 'LPL', 'LGIH', 'LGL', 'LI', 'LBGJ', 'LNNGY', 'LIANY', 'LBRDA', 'LBRDB', 'LBRDK', 'LBRDP', 'LDDFF', 'LBYE', 'LBRT', 'LBTYA', 'LBTYB', 'LBTYK', 'LGDTF', 'LILA', 'LILAK', 'FWONA', 'LLYVA', 'FWONB', 'LLYVB', 'FWONK', 'LLYVK', 'LBNW', 'LBSR', 'LICN', 'LICT', 'LICYQ', 'LFCBY', 'LFABF', 'LTGHY', 'LINS', 'LTH', 'LIF', 'LFCR', 'LFSWF', 'LLBO', 'LCTC', 'LFMD', 'LFMDP', 'LQWC', 'LTCP', 'LFST', 'LCUT', 'LFVN', 'LFWD', 'LWAY', 'LZM', 'LIFFF', 'LIGA', 'LGND', 'LGNZZ', 'LNW', 'OHCFF', 'LGSXY', 'LTBR', 'LHGI', 'LITB', 'ZEVY', 'LPTH', 'KORE', 'KOREF', 'KEP', 'KFY', 'KRNT', 'KRRO', 'KRMD', 'KRYXF', 'KSRYY', 'KOS', 'KOSS', 'KPTSF', 'KROEF', 'KHC', 'KBLB', 'UUSAF', 'KRKNF', 'KTOS', 'DNUT', 'JPX', 'KRNTY', 'KRYAY', 'KRYPF', 'KRYPY', 'KKOYY', 'KSSRF', 'KSTBF', 'KG', 'KDP', 'KEQU', 'KEWL', 'KTCC', 'KYCCF', 'KEYUF', 'KEYS', 'KZR', 'KFRC', 'KGHPF', 'KOGMF', 'KHDHF', 'KHRNF', 'KDOZF', 'JYNT', 'JBFCF', 'JBFCY', 'JLL', 'JSDA', 'JSHG', 'JRNGF', 'DERM', 'JWEL', 'JOYY', 'JPEX', 'JRJRQ', 'JRSS', 'JFWV', 'JTCPF', 'JTNB', 'JFIL', 'JUBPF', 'JSCIF', 'JUGRF', 'JBAXY', 'JBARF', 'JUMT', 'JMIA', 'JGHAF', 'JUVF', 'JPPYY', 'JMXXF', 'JUNS', 'JUSHF', 'TKAYF', 'JTKWY', 'JUTOY', 'JUVAF', 'JVTSF', 'JVSAU', 'JVSA', 'JXG', 'JZRIF', 'KPLUF', 'KPLUY', 'KTGDF', 'WDFCF', 'KNTNF', 'KANP', 'KAI', 'KNNNF', 'KAIFF', 'JPHLF', 'KATX', 'IPT', 'KMPB', 'KMT', 'KW', 'KEN', 'KLDCF', 'KFFB', 'KPELF', 'KPELY', 'KPLIF', 'KREVF', 'PPRUF', 'PPRUY', 'KMDRF', 'KARNF', 'KROS', 'ITOS', 'ITRM', 'ITEX', 'ITMPF', 'ITOCF', 'ITOCY', 'IKTO', 'ITNS', 'ITRI', 'ITRO', 'ITT', 'ITRN', 'ITVPY', 'IE', 'IVPAF', 'IVCGF', 'IVDA', 'IWAL', 'IXAQF', 'IZEA', 'IZNN', 'IZOZF', 'JSNSF', 'JSAIY', 'JJSF', 'BWEL', 'JILL', 'JAGL', 'JACK', 'JPOTF', 'JKSM', 'JXN', 'JACO', 'JADA', 'JBIO', 'MCKRF', 'TNSTF', 'JDSEF', 'JAGX', 'JAGGF', 'JAKK', 'JHX', 'JRVR', 'JAMF', 'JWLLF', 'JAMN', 'JANL', 'GDYMF', 'JHG', 'JBI', 'JANX', 'JPNRF', 'JAPSY', 'JNNDF', 'OSCUF', 'JPXGY', 'JFTH', 'JGLDF', 'JPTXF', 'JPSTF', 'JPPHY', 'JPSWY', 'JAPAF', 'JAPAY', 'JCYGY', 'JARLF', 'JMHLY', 'JSPR', 'JWSUF', 'JWSMF', 'JWSWF', 'JDNRF', 'JYD', 'JAZZ', 'JBZY', 'JBDI', 'JBGS', 'JBTC', 'JBTM', 'JCDXY', 'JCRRF', 'JDVB', 'JDDSF', 'JDSPY', 'JDCMF', 'JD', 'JDEPY', 'JCSE', 'JEF', 'JFBC', 'JFBR', 'JELD', 'JNPKF', 'JRSH', 'JROOF', 'JRONY', 'JTAI', 'DRTGF', 'JTBK', 'JBLU', 'JCTC', 'JFEEF', 'FROG', 'JGSMY', 'JGCCY', 'JDZG', 'JELCF', 'JMPLF', 'JMPLY', 'JOUT', 'INCR', 'IDCC', 'TILE', 'IFSPF', 'INTG', 'LINK', 'ITMSF', 'ICGUF', 'IBOC', 'IBATF', 'BABWF', 'ICAGY', 'INCC', 'ICTEF', 'ILDO', 'IDIG', 'IDVV', 'IFRTF', 'IGT', 'IGIC', 'INIS', 'ILAL', 'ILHMF', 'IMAQ', 'IMXI', 'INPAP', 'IPCFF', 'IPWG', 'URANF', 'INSW', 'ILST', 'ISCO', 'THM', 'IWGFF', 'IIJIY', 'IDXG', 'IPAR', 'ITRX', 'IIPZF', 'IRCKF', 'IKTSF', 'IKTSY', 'IITSF', 'ISNPY', 'INTT', 'ITOR', 'INXSF', 'IMTCF', 'IPI', 'INJJQ', 'ITJTQ', 'INTZ', 'LUNR', 'INUV', 'IVF', 'IVR', 'MHIVF', 'ISTR', 'IVCBF', 'IVTJF', 'IVSXF', 'IVSBF', 'ITIC', 'INVU', 'INVUP', 'IVCTF', 'IESVF', 'IVRO', 'IVVD', 'IRMD', 'IRTC', 'IRIX', 'IRDM', 'RSHPF', 'IRME', 'IRBT', 'IROH', 'IRWD', 'IRS', 'IRVRF', 'ISBA', 'ISOU', 'ISON', 'ISPC', 'ISPR', 'ISCNF', 'ISRL', 'ISRLU', 'ISDAY', 'ISUNQ', 'ISUZF', 'ISUZY', 'ITP', 'MBCF', 'ITGGF', 'IFBC', 'ITUB', 'INLB', 'IDEXY', 'IDCBF', 'IDCBY', 'ILPT', 'IPOAF', 'INEOF', 'IFCNF', 'IFNNF', 'IFNNY', 'CDTAF', 'INFT', 'INFIQ', 'IFRX', 'AUCUF', 'IFBD', 'IFJPY', 'INFA', 'IRMTF', 'III', 'INFY', 'IFSUF', 'IFXY', 'INFU', 'INGVF', 'ING', 'INGEF', 'NGVT', 'IMKTA', 'INGM', 'INGR', 'IKT', 'INTI', 'INBX', 'INRE', 'INM', 'INMD', 'INMB', 'IPHYF', 'IPHA', 'INEO', 'INGXF', 'INND', 'INHD', 'INNPF', 'INOD', 'IOSP', 'MPEG', 'INNV', 'IVBT', 'IVDN', 'LUCY', 'LUCYW', 'IVFH', 'IHAI', 'IIPR', 'IMTH', 'IPSI', 'ISSC', 'IVBXF', 'IVRN', 'INVX', 'INVA', 'INVZ', 'INV', 'INGN', 'NOTV', 'IVREF', 'INO', 'INZY', 'IPXHF', 'IPXHY', 'IPOOF', 'INPOY', 'IBTN', 'INSG', 'NSIT', 'INSM', 'NSP', 'IINN', 'ISPO', 'ISPOW', 'INSP', 'INSSF', 'IVP', 'INSE', 'NSPR', 'IBP', 'IIIN', 'TIL', 'MRES', 'IAUGY', 'IFCZF', 'INTA', 'ICG', 'ITGR', 'IART', 'ITRG', 'INTE', 'IAS', 'ITGLF', 'INBP', 'IGPK', 'IGCRF', 'IMTE', 'IRRX', 'INTV', 'WELNF', 'INTC', 'KASHF', 'NTLA', 'IDN', 'INBS', 'INTJ', 'ILAG', 'IPM', 'INLX', 'INTS', 'INTR', 'IHGP', 'IBKR', 'TRNR', 'ITVI', 'IESFY', 'ICHGF', 'IHG', 'IFS', 'IBDSF', 'IBDRY', 'IBEX', 'IBIDF', 'IBIO', 'IBXS', 'IBTA', 'IBWC', 'ICABY', 'ICAD', 'IEP', 'ICCM', 'ICFI', 'ICHR', 'IBN', 'ICL', 'ICON', 'ICNM', 'ICLR', 'ICNB', 'ICMFF', 'ICCT', 'ICTSF', 'ICUI', 'IZM', 'IDA', 'COPR', 'IDR', 'IDGR', 'IPWR', 'IDEXQ', 'IDYA', 'IDKOF', 'IDKOY', 'IDTA', 'INVE', 'IEX', 'IDGC', 'IDT', 'IDWM', 'IEHC', 'IESC', 'IROQ', 'IFAN', 'IFMK', 'IGGHY', 'IGPTF', 'IGC', 'IGEN', 'IGGGF', 'IGMS', 'IGIFF', 'IPGDF', 'IIDDY', 'IHRT', 'IHICY', 'IHS', 'IH', 'ITOX', 'IJJP', 'IKNA', 'ILIM', 'ILIKF', 'ILLMF', 'ILMN', 'ILKAF', 'ILKAY', 'ILUS', 'IMCC', 'IMAB', 'BACK', 'IFLXF', 'IMTL', 'IAGX', 'IWSY', 'ARXRF', 'IPNFF', 'IMXCF', 'IMAX', 'IMCDY', 'IMRFF', 'IMIAF', 'IMIUY', 'IMTX', 'IMMR', 'IMMX', 'ICCC', 'IMRX', 'IMUX', 'IBRX', 'IMNM', 'IPA', 'IMVT', 'IMRN', 'IHLDF', 'IMPM', 'IBO', 'IFUS', 'ISVLF', 'IMPUF', 'IMPUY', 'IPXAF', 'IMBBF', 'IMBBY', 'IPMLF', 'IMO', 'IMPP', 'IMPPP', 'IPRC', 'IPGGF', 'PI', 'IUGNF', 'IMNN', 'IMVIF', 'INOH', 'IVME', 'INAB', 'INBC', 'IXHL', 'IGTA', 'IOR', 'INDP', 'IEGCF', 'IRT', 'INDB', 'IBCP', 'IFLM', 'INDI', 'INDV', 'IGEX', 'ITAYY', 'IDCN', 'PIFMY', 'INDO', 'INQD', 'PTITF', 'ISMAF', 'ISMAY', 'IDEXF', 'HTLM', 'HMST', 'HWIN', 'HTB', 'HNHPF', 'HONT', 'HNDAF', 'HMC', 'HNST', 'HBEIF', 'HOKCY', 'HKSHF', 'HKXCF', 'HKXCY', 'HKTVY', 'HGYN', 'HCIL', 'HKHGF', 'HNGKY', 'HLP', 'HOFT', 'HOOK', 'HOPE', 'HPNN', 'HPTO', 'HMN', 'HBNC', 'HNCUF', 'HGPI', 'HKHC', 'HZNFF', 'HSPOU', 'HSPO', 'HSPTU', 'HRZN', 'HTFB', 'HTFC', 'HRL', 'HHLKF', 'HOTH', 'HLI', 'HOUR', 'HUSA', 'HOV', 'HOVNP', 'HHH', 'HWDJF', 'HOCPF', 'HOCPY', 'HPIL', 'KICK', 'HPQFF', 'HQGE', 'HBCYF', 'HSBC', 'HSTC', 'HTGMQ', 'HUDI', 'HHEGF', 'HUNGF', 'HUBC', 'HUBG', 'HUBS', 'HBM', 'HSON', 'HPP', 'HDSN', 'HUGPF', 'BOSSY', 'HGTXU', 'HUHU', 'HUMT', 'HMBL', 'HUML', 'HBANM', 'HBANP', 'HBANL', 'HUN', 'BBBMF', 'HURC', 'HURN', 'HSQVY', 'HUT', 'HUPHY', 'HUTCY', 'HCM', 'HUYA', 'HWGG', 'HWH', 'H', 'HPSIF', 'HYMC', 'HYMCL', 'HYDTF', 'HRNNF', 'PYBX', 'HYFM', 'HYEG', 'HUIPF', 'HGRAF', 'HYLN']
len(stocks)

816

In [ ]:
stocks = ["H"]

In [ ]:
#SECOND BATCH OF STOCKS, CAN BE CHANGED ['LAZ', 'LZB', 'LCII', 'LCNB', 'LEMIF', 'LBUY', 'LFLY', 'LPTX', 'LEA', 'LEAT', 'LEE', 'LEEEF', 'LEGIF', 'LGCY', 'LEGH', 'LZ', 'LEGT', 'LEGN', 'LPSIF', 'LEG', 'LEBGF', 'LGRDY', 'LDOS', 'LMAT', 'LMND', 'LC', 'TREE', 'LEN', 'LII', 'LNSR', 'LENZ', 'LNZNF', 'LECRF', 'DRS', 'LEFUF', 'LYSFY', 'LESL', 'LEVI', 'LVXFF', 'LEXX', 'LXEO', 'LXRX', 'LIFD', 'LGIH', 'LGL', 'LI', 'LBRDA', 'LBRDB', 'LBRDK', 'LBRDP', 'LDDFF', 'LBRT', 'LBTYA', 'LBTYB', 'LBTYK', 'LGDTF', 'LILA', 'LILAK', 'FWONA', 'LLYVA', 'FWONB', 'LLYVB', 'FWONK', 'LLYVK', 'LBSR', 'LFCBY', 'LFABF', 'LTH', 'LIF', 'LFCR', 'LFSWF', 'LCTC', 'LFMD', 'LFMDP', 'LFST', 'LCUT', 'LFVN', 'LFWD', 'LWAY', 'LIFFF', 'LGND', 'LNW', 'OHCFF', 'LGSXY', 'LTBR', 'LPTH', 'KORE', 'KEP', 'KFY', 'KRNT', 'KRRO', 'KRMD', 'KRYXF', 'KOS', 'KPTSF', 'KHC', 'KBLB', 'UUSAF', 'KRKNF', 'KTOS', 'DNUT', 'KR', 'KRNTY', 'KKOYY', 'KSSRF', 'KSTBF', 'KDP', 'KTCC', 'KEY', 'KEYUF', 'KEYS', 'KZR', 'KFRC', 'KGHPF', 'KOGMF', 'KDOZF', 'JYNT', 'JBFCF', 'JBFCY', 'JLL', 'JRNGF', 'DERM', 'JPM', 'JFIL', 'JUGRF', 'JMIA', 'JUVF', 'JNPR', 'JUNS', 'JUSHF', 'JVSAU', 'JVSA', 'JZRIF', 'KPLUF', 'KPLUY', 'KTGDF', 'WDFCF', 'KNTNF', 'KANP', 'KAI', 'KATX', 'KMT', 'KW', 'KLDCF', 'KFFB', 'KVUE', 'KMDRF', 'KARNF', 'KROS', 'ITOS', 'ITRM', 'ITRI', 'ITT', 'ITRN', 'IE', 'IVPAF', 'IVCGF', 'IWAL', 'IXAQF', 'IZEA', 'IZOZF', 'JJSF', 'JBHT', 'JILL', 'SJM', 'JKHY', 'JXN', 'J', 'MCKRF', 'TNSTF', 'JAGX', 'JAGGF', 'JAKK', 'JHX', 'JRVR', 'JAMF', 'JWLLF', 'JHG', 'JBI', 'JANX', 'JGLDF', 'JSPR', 'JDNRF', 'JAZZ', 'JBGS', 'JBTM', 'JDCMF', 'JD', 'JEF', 'JELD', 'JNPKF', 'JRSH', 'JROOF', 'JRONY', 'JTAI', 'JBLU', 'FROG', 'JGSMY', 'JBL', 'JCI', 'JOUT', 'IDCC', 'TILE', 'IFSPF', 'ITMSF', 'IBATF', 'IBM', 'ICTEF', 'IFF', 'IFRTF', 'IGT', 'IMAQ', 'IMXI', 'IP', 'INPAP', 'IPCFF', 'URANF', 'INSW', 'THM', 'IIJIY', 'IPAR', 'IPG', 'IIPZF', 'IRCKF', 'INTT', 'ITOR', 'INXSF', 'IMTCF', 'IPI', 'INJJQ', 'ITJTQ', 'INTZ', 'INTU', 'LUNR', 'ISRG', 'INUV', 'IVF', 'IVZ', 'IVR', 'MHIVF', 'ISTR', 'IVSXF', 'IVSBF', 'INVU', 'INVUP', 'INVH', 'IVVD', 'IRMD', 'IRTC', 'IRDM', 'IRME', 'IRBT', 'IROH', 'IRM', 'IRWD', 'IRS', 'IRVRF', 'ISBA', 'ISOU', 'ISPC', 'ISPR', 'ISRL', 'ISRLU', 'ISDAY', 'MBCF', 'ITUB', 'IDCBF', 'IDCBY', 'ILPT', 'IPOAF', 'INEOF', 'IFNNF', 'IFNNY', 'CDTAF', 'IFRX', 'AUCUF', 'INFA', 'IRMTF', 'III', 'IFSUF', 'INFU', 'IR', 'NGVT', 'INGM', 'INGR', 'IKT', 'INTI', 'INBX', 'INRE', 'INM', 'INMD', 'INMB', 'INGXF', 'INHD', 'INNPF', 'INOD', 'IOSP', 'INNV', 'IVDN', 'LUCY', 'LUCYW', 'IVFH', 'IIPR', 'IMTH', 'IPSI', 'ISSC', 'INVA', 'INVZ', 'INV', 'INGN', 'NOTV', 'IVREF', 'INO', 'INZY', 'IPOOF', 'INPOY', 'INSG', 'NSIT', 'INSM', 'NSP', 'ISPO', 'ISPOW', 'INSP', 'IVP', 'INSE', 'NSPR', 'IBP', 'IIIN', 'TIL', 'PODD', 'IFCZF', 'INTA', 'ITGR', 'IART', 'ITRG', 'IAS', 'INBP', 'IGCRF', 'IRRX', 'INTV', 'WELNF', 'INTC', 'KASHF', 'NTLA', 'IDN', 'INBS', 'IPM', 'INLX', 'INTS', 'INTR', 'IBKR', 'ICE', 'IFS', 'IBEX', 'IBIO', 'IBTA', 'ICAD', 'IEP', 'ICCM', 'ICFI', 'ICHR', 'ICL', 'ICON', 'ICLR', 'ICMFF', 'ICCT', 'ICUI', 'IDA', 'COPR', 'IDR', 'IPWR', 'IDYA', 'IDTA', 'INVE', 'IEX', 'IDXX', 'IDT', 'IEHC', 'IESC', 'IROQ', 'IGC', 'IGMS', 'IGIFF', 'IHRT', 'IHS', 'ITOX', 'IKNA', 'ITW', 'ILLMF', 'ILMN', 'IMCC', 'IFLXF', 'ARXRF', 'IPNFF', 'IMAX', 'IMRFF', 'IMTX', 'IMMX', 'IMRX', 'IMUX', 'IBRX', 'IMNM', 'IPA', 'IMVT', 'IHLDF', 'IBO', 'ISVLF', 'IPMLF', 'IMO', 'IMPP', 'IMPPP', 'PI', 'IMNN', 'INAB', 'IXHL', 'IGTA', 'IOR', 'INCY', 'INDP', 'IEGCF', 'IRT', 'INDB', 'IBCP', 'INDI', 'INDV', 'ITAYY', 'PIFMY', 'PTITF', 'ISMAF', 'ISMAY', 'HMST', 'HTB', 'HNHPF', 'HNDAF', 'HMC', 'HNST', 'HBEIF', 'HON', 'HGYN', 'HCIL', 'HOFT', 'HOPE', 'HMN', 'HBNC', 'HNCUF', 'HKHC', 'HSPOU', 'HSPO', 'HRZN', 'HRL', 'HST', 'HOTH', 'HLI', 'HOV', 'HOVNP', 'HHH', 'HWM', 'HPQ', 'HPQFF', 'HUNGF', 'HUBG', 'HUBB', 'HUBS', 'HBM', 'HSON', 'HPP', 'HDSN', 'HUGPF', 'BOSSY', 'HMBL', 'HBAN', 'HBANM', 'HBANP', 'HBANL', 'HII', 'HUN', 'BBBMF', 'HURN', 'HSQVY', 'HUT', 'HWH', 'H', 'HPSIF', 'HYMC', 'HYMCL', 'HYDTF', 'HRNNF', 'HYFM', 'HGRAF', 'HYLN', 'HUM']
print(stocks)

['LAZ', 'LZB', 'LCNB', 'LFLY', 'LPTX', 'LEA', 'LEGH', 'LZ', 'LEGN', 'LEG', 'LGRDY', 'LDOS', 'LMAT', 'LMND', 'LC', 'TREE', 'LEN', 'LII', 'LNSR', 'LESL', 'LEVI', 'LEXX', 'LXEO', 'LXRX', 'LGIH', 'LGL', 'LI', 'LBRT', 'LBTYA', 'LILA', 'LILAK', 'FWONA', 'FWONK', 'LTH', 'LIF', 'LFMD', 'LCUT', 'LFWD', 'LWAY', 'LGND', 'LNW', 'KRNT', 'KRMD', 'KOS', 'KHC', 'KTOS', 'KR', 'KDP', 'KEY', 'KEYS', 'KFRC', 'JYNT', 'JLL', 'DERM', 'JPM', 'JNPR', 'JUSHF', 'KAI', 'KW', 'KVUE', 'KROS', 'ITOS', 'ITRM', 'ITT', 'ITRN', 'IVPAF', 'IZEA', 'JJSF', 'JBHT', 'JILL', 'SJM', 'JXN', 'J', 'JAGX', 'JAGGF', 'JAKK', 'JRVR', 'JAMF', 'JHG', 'JBI', 'JANX', 'JSPR', 'JAZZ', 'JBTM', 'JD', 'JEF', 'JELD', 'JRSH', 'JRONY', 'JBLU', 'FROG', 'JBL', 'JCI', 'IBM', 'IFF', 'IGT', 'IMXI', 'IP', 'INSW', 'IPAR', 'INTT', 'IPI', 'INTZ', 'LUNR', 'ISRG', 'INUV', 'IVZ', 'IVR', 'ISTR', 'IRMD', 'IRDM', 'IRME', 'IRM', 'IRWD', 'ISBA', 'ISPC', 'ITUB', 'INFA', 'INFU', 'IR', 'NGVT', 'INGR', 'IKT', 'INMD', 'INMB', 'INOD', 'IOSP', 'LUCY', 'IIPR', 'INVA', 'I

In [ ]:
#X = pd.DataFrame({"":predictions})
y = pd.DataFrame({"":label})


cr = classification_report(X,y)
print(cr)

cm = confusion_matrix(X,y)
print(cm)

              precision    recall  f1-score   support

           0       0.86      0.50      0.63       616
           1       0.22      0.64      0.33       136

    accuracy                           0.52       752
   macro avg       0.54      0.57      0.48       752
weighted avg       0.75      0.52      0.58       752

[[306 310]
 [ 49  87]]


In [ ]:
X = pd.read_excel("/content/asdf-1.xlsx")
X = X.iloc[:,1]

In [ ]:
df = pd.DataFrame({"y":label})
df.shape

(966, 1)

In [ ]:
for i in error:
  p = stocks.index(i)
  print(p)

115
141
210
399


In [ ]:
df.to_excel("predictions_labels.xlsx")

In [ ]:
X.to_excel("asdf.xlsx")

In [ ]:
df = pd.read_csv("/content/Unseen_Data_Balanced_Answer.csv")
stock_list = df["stock"].tolist()
print(stock_list)
len(stock_list)

['JYNT', 'HST', 'LBGJ', 'LICN', 'INPAP', 'LXRX', 'HONT', 'KMPB', 'IESVF', 'LRE', 'IFJPY', 'IDCC', 'INFA', 'IVREF', 'KORE', 'IFSPF', 'IDTA', 'KICK', 'HPSIF', 'LSE', 'IDR', 'JBFCF', 'TNSTF', 'HHLKF', 'JNPR', 'LCTC', 'INEOF', 'INDB', 'HWH', 'JFBR', 'LDDFF', 'LEGH', 'KMT', 'LFLY', 'INBC', 'HNST', 'LFGP', 'INLX', 'IDGR', 'IJJP', 'LBYE', 'KNTNF', 'LQWC', 'KSSRF', 'HOKCY', 'JBTM', 'PODD', 'INVZ', 'LBTYA', 'JUGRF', 'IRTC', 'HUBG', 'HUT', 'HSQVY', 'LEGIF', 'HUBS', 'IDGC', 'OSCUF', 'JXN', 'ILPT', 'INFU', 'HOCPY', 'IWAL', 'INVU', 'IPXHY', 'LHHMQ', 'LEE', 'LEHLQ', 'HYDTF', 'JUVAF', 'LIFD', 'ICLR', 'INTZ', 'LBRDP', 'IGPK', 'IDEXQ', 'INGN', 'IDN', 'HRL', 'III', 'INJJQ', 'IFRX', 'LUCYW', 'LZ', 'ICCM', 'KARNF', 'IBP', 'LFABF', 'IFF', 'LTH', 'KG', 'JHG', 'IPCFF', 'HDSN', 'IBDRY', 'IVSBF', 'IHS', 'WDFCF', 'LEXX', 'LIFFF', 'ITW', 'IFBC', 'ITGLF', 'ILKAY', 'DRS', 'JDSPY', 'HNGKY', 'IMVT', 'IHLDF', 'JPEX', 'INXSF', 'INTI', 'IDCBY', 'HUHU', 'HSON', 'JDSEF', 'ILMN', 'IDEXF', 'ICFI', 'HKXCF', 'ICTEF', 'JACK',

728

In [ ]:
df = pd.read_csv("/content/Final_estimator_features.csv")
y = pd.read_csv("/content/Final_Estimators_labels.csv")
df = pd.concat([df,y], axis=1)
df.shape

(1415, 13)

In [ ]:
df.to_csv("Final Estimators Training Data.csv")

In [ ]:
df = df.drop(columns=["Unnamed: 0"])
df.head()

,0,1,2,3,4,5,6,7,8,9,Unnamed: 1
0,0.043535,0.05,0.029715,0.106218,0.119203,0.607856,0.157564,0.14,0.174578,0.880797,0
1,0.710010,0.42,0.297501,0.923283,0.119203,-100.000000,-100.000000,-100.00,0.834503,-100.000000,1
2,0.866327,0.95,0.945596,0.713986,0.880797,0.502028,0.462447,0.91,0.909576,0.119203,1
3,0.897831,1.00,0.344744,0.915353,0.119203,-100.000000,-100.000000,-100.00,0.895557,-100.000000,1
4,0.897831,1.00,0.137375,0.915353,0.119203,-100.000000,-100.000000,-100.00,0.895557,-100.000000,1


In [ ]:
df = pd.read_csv("/content/Meta Test 3 (40,96).csv")
#y = df.pop("label")
#df.shape, y.shape

In [ ]:
# Separate the DataFrame into two groups based on the label
df_0 = df[df['label'] == 0]
df_1 = df[df['label'] == 1]

# Get the number of rows in the minority class
n_0 = len(df_0)
n_1 = len(df_1)

# If there are more 0s than 1s, randomly sample the 0s
if n_0 > n_1:
    df_0 = df_0.sample(n=n_1, random_state=42)

# If there are more 1s than 0s, randomly sample the 1s
elif n_1 > n_0:
    df_1 = df_1.sample(n=n_0, random_state=42)

# Concatenate the balanced DataFrame
df = pd.concat([df_0, df_1])

df = df.sample(frac=1)

In [ ]:

df.head()

,stock,label
706,HNGKY,1
5,LEMIF,1
306,JCDXY,0
98,LTH,0
565,IEP,0


In [ ]:
stocks = ['LSPD', 'LVPR', 'LWLG', 'LILMF', 'LMB', 'LVGI', 'LIMX', 'LMNR', 'LIMAF', 'LINC', 'LECO', 'LNC', 'LIND', 'LIN', 'LINIF', 'LNN', 'LNDAF', 'LCTX', 'LINE', 'LINMF', 'BOTY', 'LNMG', 'LINUF', 'PSBXP', 'PSBYP', 'PSBZP', 'LKREF', 'LGCB', 'LNKB', 'LNKS', 'LCGMF', 'LEVGQ', 'LEVWQ', 'LGHL', 'LOMLF', 'LRRIF', 'CUB', 'LION', 'LINRF', 'LIPO', 'LPCN', 'LIQT', 'YVRLF', 'LQDA', 'LQDT', 'LQMT', 'LSTA', 'LSIIF', 'LAD', 'LAC', 'LAR', 'LTMCF', 'LTUM', 'LXENF', 'IONGF', 'LTHCF', 'LOMEF', 'LITRF', 'LISMF', 'LUVSF', 'LBNKF', 'LITSD', 'LFUS', 'LIVN', 'GTREF', 'LYV', 'LOB', 'LIVE', 'LVCE', 'LVO', 'LPSN', 'RAMP', 'LVVV', 'LVWR', 'LVWD', 'LMMFF', 'LXEH', 'JSGRY', 'LIXT', 'LIXTW', 'LKQ', 'LLDTF', 'LYG', 'LMFA', 'LMPX', 'LNGNF', 'LDI', 'LOAR', 'LOBEF', 'LBLCF', 'LOBO', 'LCFY', 'LOCL', 'LOCLW', 'LOCM', 'LZRFY', 'LBAS', 'LMT', 'L', 'LOECF', 'LRFC', 'LGMK', 'LPA', 'LOGI', 'LOMA', 'LMRMF', 'LONCF', 'LDNXF', 'LNSTY', 'PPLFY', 'LNSPF', 'LGVN', 'XAGE', 'LFIN', 'LGFRY', 'LZAGF', 'LZAGY', 'LOOP', 'LPTV', 'LRLCF', 'LRLCY', 'LSANF', 'LOTE', 'LTRY', 'LTUS', 'LTSRF', 'LOT', 'LPX']

df = df_meta.iloc[0].to_frame().T
df["label"] = 0

print(df.shape)

for i in range(len(stocks)):
  try:
    successful, failed, balancesheet, cashflow, analyst_target_eps, incomestmt, earnings_est, earnings_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta = transformed_testing_data(stocks[i])
    try:
      change = (float(yf.Ticker(stocks[i]).history(start = "2025-06-01", end = "2025-06-04")["Close"].tail(1).iloc[0]) - float(yf.Ticker(stocks[i]).history(start = "2025-03-25", end = "2025-03-31")["Close"].tail(1).iloc[0]))/float(yf.Ticker(stocks[i]).history(start = "2025-03-25", end = "2025-03-31")["Close"].tail(1).iloc[0])
      print(i, stocks[i], change)
      if change > 0:
          meta["label"] = 1
      elif change == 0:
        pass
      else:
          meta["label"] = 0
      df = pd.concat([df, meta])
    except:
      print(stocks[i], " failed")
  except:
    print("🔴", stocks[i])


(1, 96)
0 LSPD 0.215621562733304
yfinance.Ticker object <LVPR> balancesheet failed to load
yfinance.Ticker object <LVPR> cashflow failed to load


ERROR:yfinance:HTTP Error 404: 
ERROR:yfinance:HTTP Error 404: 


yfinance.Ticker object <LVPR> eps trend failed to load
yfinance.Ticker object <LVPR> incomestmt failed to load
yfinance.Ticker object <LVPR> earnings estimates failed to load


ERROR:yfinance:HTTP Error 404: 
ERROR:yfinance:$LVPR: possibly delisted; no price data found  (1d 2025-06-01 -> 2025-06-04)


yfinance.Ticker object <LVPR> earnings history failed to load
yfinance.Ticker object <LVPR> revenue estimates failed to load
LVPR  failed
yfinance.Ticker object <LWLG> earnings history failed to load
🔴 LWLG
yfinance.Ticker object <LILMF> balancesheet failed to load
yfinance.Ticker object <LILMF> cashflow failed to load
yfinance.Ticker object <LILMF> incomestmt failed to load
yfinance.Ticker object <LILMF> earnings history failed to load
🔴 LILMF
4 LMB 0.7698662134783766
yfinance.Ticker object <LVGI> balancesheet failed to load
yfinance.Ticker object <LVGI> cashflow failed to load
yfinance.Ticker object <LVGI> incomestmt failed to load
yfinance.Ticker object <LVGI> earnings history failed to load
5 LVGI 0.33333323632056977
yfinance.Ticker object <LIMX> incomestmt failed to load
yfinance.Ticker object <LIMX> earnings history failed to load
6 LIMX -0.6025333404541016
7 LMNR -0.09579634205518742
yfinance.Ticker object <LIMAF> earnings history failed to load
8 LIMAF 0.3284229664175507
9 LINC

...

yfinance.Ticker object <LEVWQ> earnings history failed to load
yfinance.Ticker object <LEVWQ> revenue estimates failed to load
32 LEVWQ 0.0
yfinance.Ticker object <LGHL> balancesheet failed to load
yfinance.Ticker object <LGHL> cashflow failed to load
yfinance.Ticker object <LGHL> earnings history failed to load
33 LGHL -0.18857145309448242
yfinance.Ticker object <LOMLF> incomestmt failed to load
yfinance.Ticker object <LOMLF> earnings history failed to load
34 LOMLF -0.04347828340440215
yfinance.Ticker object <LRRIF> earnings history failed to load
35 LRRIF 0.14210523921697063
yfinance.Ticker object <CUB> cashflow failed to load
yfinance.Ticker object <CUB> incomestmt failed to load
yfinance.Ticker object <CUB> earnings history failed to load
36 CUB 0.027343817666407325
yfinance.Ticker object <LION> balancesheet failed to load
yfinance.Ticker object <LION> cashflow failed to load
yfinance.Ticker object <LION> incomestmt failed to load
37 LION -0.0201883744274344
yfinance.Ticker object

ERROR:yfinance:HTTP Error 404: 


yfinance.Ticker object <LOCLW> eps trend failed to load
yfinance.Ticker object <LOCLW> earnings estimates failed to load


ERROR:yfinance:HTTP Error 404: 


yfinance.Ticker object <LOCLW> earnings history failed to load
yfinance.Ticker object <LOCLW> revenue estimates failed to load
93 LOCLW 0.0
yfinance.Ticker object <LOCM> balancesheet failed to load
yfinance.Ticker object <LOCM> cashflow failed to load
yfinance.Ticker object <LOCM> incomestmt failed to load
🔴 LOCM
yfinance.Ticker object <LZRFY> earnings history failed to load
95 LZRFY 0.2414355614100129
yfinance.Ticker object <LBAS> balancesheet failed to load
yfinance.Ticker object <LBAS> cashflow failed to load
yfinance.Ticker object <LBAS> incomestmt failed to load
yfinance.Ticker object <LBAS> earnings history failed to load
96 LBAS -0.033942573508067765
97 LMT 0.0951041413785961
yfinance.Ticker object <L> earnings history failed to load
98 L -0.010553435855553121
yfinance.Ticker object <LOECF> incomestmt failed to load
yfinance.Ticker object <LOECF> earnings history failed to load
99 LOECF -0.036796578125818435
100 LRFC -0.17832173049559719
101 LGMK -0.8399999947845936
yfinance.Tic

#The Model


In [ ]:
#These are all the S&P500 stocks
stocks= ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'CAG', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'FI', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEV', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'IPG', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'K', 'KVUE', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LW', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MTCH', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PARA', 'PH', 'PAYX', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SOLV', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VLTO', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WBA', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']
len(stocks)

503

In [ ]:
# no earnings 28th
stocks = ['ACN', 'AZO', 'KMX', 'CCL', 'CTAS', 'CAG', 'STZ', 'CPRT', 'COST', 'DRI', 'FDS', 'JBL', 'LW', 'LEN', 'MKC', 'MU', 'NKE', 'PAYX', 'WBA']
len(stocks)

20

In [ ]:
l = []
l2 = []
for i in range(len(stocks)):
  if datetime.strptime(est_date(stocks[i]), "%Y-%m-%d")>datetime.strptime("2025-09-01","%Y-%m-%d"):
    print(stocks[i], i)
    l.append(stocks[i])
  else:
    l2.append(stocks[i])

len(l)

ADBE 1


1

In [ ]:
stocks = l
print(len(stocks),stocks)

1 ['ADBE']


In [ ]:
l = ['MMM', 'AOS', 'ABT', 'ABBV', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APTV', 'ACGL', 'ADM', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AVB', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BK', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BLDR', 'BG', 'BXP', 'CHRW', 'CDNS', 'CZR', 'CPT', 'CPB', 'COF', 'CAH', 'CARR', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'CINF', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COIN', 'CL', 'CMCSA', 'COP', 'ED', 'CEG', 'COO', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'CTRA', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DDOG', 'DVA', 'DAY', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'ELV', 'EMR', 'ENPH', 'ETR', 'EOG', 'EPAM', 'EQT', 'EFX', 'EQIX', 'EQR', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FICO', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FSLR', 'FE', 'FI', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEV', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'INCY', 'IR', 'PODD', 'INTC', 'ICE', 'IFF', 'IP', 'IPG', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'K', 'KVUE', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LVS', 'LDOS', 'LII', 'LLY', 'LIN', 'LYV', 'LKQ', 'LMT', 'L', 'LOW', 'LULU', 'LYB', 'MTB', 'MPC', 'MKTX', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MTCH', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MSFT', 'MAA', 'MRNA', 'MHK', 'MOH', 'TAP', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PARA', 'PH', 'PAYC', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'POOL', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'RL', 'RJF', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SOLV', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TTD', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VTR', 'VLTO', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VTRS', 'VICI', 'V', 'VST', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']



In [ ]:
for i in l:
  stocks.remove(i)
len(stocks)

20

In [ ]:
print(stocks)

['ACN', 'ADBE', 'AZO', 'KMX', 'CCL', 'CTAS', 'CAG', 'STZ', 'CPRT', 'COST', 'DRI', 'FDS', 'JBL', 'LW', 'LEN', 'MKC', 'MU', 'NKE', 'PAYX', 'WBA']


In [ ]:
bull_stocks = []
bull_stocks_proba = []
bear_stocks = []
bear_stocks_proba = []

bullish_stocks = []
bearish_stocks = []
uncertain_stocks = []

for i in range(len(stocks)):
  prediction, proba = fundemental_model(stocks[i])

  if proba > 0.91:
    bull_stocks_proba.append(proba)
    bull_stocks.append(stocks[i])
  if proba < 0.1:
    bear_stocks_proba.append(proba)
    bear_stocks.append(stocks[i])
  if proba > 0.8:
    bullish_stocks.append(stocks[i])
  if proba < 0.15:
    bearish_stocks.append(stocks[i])

  else:
    uncertain_stocks.append(stocks[i])

for i in bull_stocks_proba:
  print(i)


  ADBE
  Successful: 10
        ['balancesheet', 'cashflow', 'analyst_target_eps', 'incomestmt', 'earnings_est', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'rev_est', 'meta']
  Failed: 0
        []
  
🔹 1 🔹 ~83% accuracy 🔹
0.9521009321832054


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


In [ ]:
len(bull_stocks), len(bear_stocks), len(bullish_stocks), len(bearish_stocks)

(132, 74, 231, 119)

In [ ]:
print(f'''
Bull Stocks (91%+ Bull Confidence): {bull_stocks}
Bear Stocks (10%- Bull Confidence): {bear_stocks}
Bullish Stocks (80-91% Bull Confidence): {bullish_stocks}
Bearish Stocks (10-15% Bull Confidence): {bearish_stocks}
''')


Bull Stocks (91%+ Bull Confidence): ['MMM', 'ABT', 'AMD', 'ABNB', 'ALLE', 'ALL', 'GOOGL', 'MO', 'AXP', 'AMP', 'ADI', 'APTV', 'ADSK', 'AVY', 'BKR', 'BALL', 'BX', 'XYZ', 'BK', 'BKNG', 'BSX', 'COF', 'CBOE', 'CDW', 'COR', 'SCHW', 'CMG', 'CFG', 'CTSH', 'CEG', 'GLW', 'CTVA', 'CSX', 'DDOG', 'DAY', 'DE', 'DXCM', 'DOV', 'ECL', 'FAST', 'FIS', 'FITB', 'FSLR', 'F', 'BEN', 'FCX', 'GNRC', 'GD', 'GM', 'GPC', 'GS', 'HLT', 'HOLX', 'HON', 'HST', 'HWM', 'HUBB', 'HII', 'IBM', 'IDXX', 'IR', 'ISRG', 'IRM', 'J', 'JCI', 'JPM', 'KEY', 'KR', 'LVS', 'LDOS', 'LII', 'LIN', 'LYV', 'LMT', 'LOW', 'MTB', 'MLM', 'MCD', 'META', 'MET', 'MGM', 'MCHP', 'MSFT', 'MS', 'NTAP', 'NWSA', 'NWS', 'NDSN', 'NUE', 'ON', 'PLTR', 'PANW', 'PAYC', 'PYPL', 'PNR', 'PM', 'PNC', 'PPG', 'PTC', 'PSA', 'PWR', 'RL', 'RTX', 'RF', 'RSG', 'ROK', 'ROL', 'SBAC', 'NOW', 'STT', 'STLD', 'STE', 'SYK', 'SYF', 'TTWO', 'TPR', 'TEL', 'TDY', 'TXT', 'TJX', 'TKO', 'TT', 'TDG', 'USB', 'ULTA', 'VLO', 'VRSK', 'WAB', 'WFC', 'WDAY', 'XYL', 'ZBRA']
Bear Stocks (10%-

In [ ]:
#as of sep28, added to TAx
good_stocks = ['ADBE', 'ADSK', 'DLTR', 'PANW', 'TJX','wmt' ,'DE', 'LOW', 'NTAP', 'ULTA',"hd","wday","ndsn","adi",'ABT', 'AMD', 'ABNB', 'ALLE', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AXP', 'AIG', 'AMP', 'APTV', 'AVY', 'BALL', 'BLK', 'BX', 'XYZ', 'BK', 'BKNG', 'BSX', 'BR', 'COF', 'CAH', 'CARR', 'CBOE', 'CDW', 'COR', 'SCHW', 'CHTR', 'CMG', 'CFG', 'CTSH', 'CEG', 'GLW', 'CTVA', 'CSX', 'DDOG', 'DAY', 'DAL', 'DXCM', 'D', 'DOV', 'EXPE', 'FAST', 'FIS', 'FITB', 'FSLR', 'F', 'BEN', 'FCX', 'GE', 'GNRC', 'GD', 'GPC', 'GS', 'HCA', 'HLT', 'HOLX', 'HON', 'HST', 'HWM', 'HUBB', 'HII', 'IBM', 'IDXX', 'IR', 'ISRG', 'IRM', 'J', 'JCI', 'JPM', 'KEY', 'KKR', 'LVS', 'LDOS', 'LII', 'LIN', 'LYV', 'LMT', 'MTB', 'MLM', 'MA', 'MCD', 'META', 'MET', 'MGM', 'MCHP', 'MSFT', 'MCO', 'MS', 'NWSA', 'NWS', 'NRG', 'NUE', 'ON', 'PLTR', 'PAYC', 'PYPL', 'PNR', 'PM', 'PNC', 'PPG', 'PTC', 'PSA', 'PWR', 'RL', 'RTX', 'RF', 'RSG', 'ROK', 'ROL', 'SBAC', 'NOW', 'STT', 'STLD', 'STE', 'SYK', 'SYF', 'TTWO', 'TEL', 'TDY', 'TXT', 'TKO', 'TDG', 'USB', 'VLO', 'VRSK', 'WAB', 'WFC', 'XYL', 'ZBRA']
len(good_stocks)

145

In [ ]:
stock = input("Enter stock symbol: ")
prediction, proba = fundemental_model(stock)

print(f''' {prediction}

⛋ Prediction made for {est_date(stock)} ⛋


⛋ Predicted Probabilities:  ⛋
               Bull: {proba},
               Bear: {1-proba}


''')

Enter stock symbol: hd

  hd
  Successful: 10
        ['balancesheet', 'cashflow', 'analyst_target_eps', 'incomestmt', 'earnings_est', 'earnings_hist', 'earnings_hist_est', 'earnings_hist_est_eps', 'rev_est', 'meta']
  Failed: 0
        []
  
🔹 1 🔹 ~69% accuracy 🔹
 1

⛋ Prediction made for 2025-10-31 ⛋


⛋ Predicted Probabilities:  ⛋
               Bull: 0.8593330493237588,
               Bear: 0.14066695067624124





/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ExtraTreesClassifier was fitted with feature names
  warnings.warn(


#Debugging


In [ ]:
stock = "aapl"

weighted_prediction = []
predictions = []

successful, failed, balancesheet, cashflow, analyst_target_eps, incomestmt, earnings_est, earnings_hist, earnings_hist_est, earnings_hist_est_eps, rev_est, meta = transformed_testing_data(stock)
print(f'''
{stock}
Successful: {len(successful)}
    {successful}
Failed: {len(failed)}
    {failed}
''')
if len(failed)==10:
  print("🔺🔺🔺 error occured 🔺🔺🔺")
  return f"error with {stock}",f"error with {stock}",f"error with {stock}",f"error with {stock}"

else:
#if "balancesheet" in successful:
  try:
      pred = clf_balancesheet.predict_proba(balancesheet)
      #weighted_prediction.append(pred[0,1]*(3025/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3025/32906))
      predictions.append(-100)
      print(f"Error occurred with balancesheet: {e}")

  #if "cashflow" in successful:
  try:
      pred = clf_cashflow.predict_proba(cashflow)
      #weighted_prediction.append(pred[0,1]*(3025/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3025/32906))
      predictions.append(-100)
      print(f"Error occurred with cashflow: {e}")

  #if "analyst_target_eps" in successful:
  try:
      pred = clf_analystpricetarget_eps.predict_proba(analyst_target_eps)
      #weighted_prediction.append(pred[0,1]*(3025/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3025/32906))
      predictions.append(-100)
      print(f"Error occurred with analyst_target_eps: {e}")

  #if "incomestmt" in successful:
  try:
      pred = clf_income_statement.predict_proba(incomestmt)
      #weighted_prediction.append(pred[0,1]*(3481/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3481/32906))
      predictions.append(-100)
      print(f"Error occurred with incomestmt: {e}")

  #if "earnings_est" in successful:
  try:
      pred = clf_earnings_est.predict_proba(earnings_est)
      #weighted_prediction.append(pred[0,1]*(2916/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(2916/32906))
      predictions.append(-100)
      print(f"Error occurred with earnings_est: {e}")

  #if "earnings_hist" in successful:
  try:
      pred = clf_earnings_hist.predict_proba(earnings_hist)
      #weighted_prediction.append(pred[0,1]*(3600/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3600/32906))
      predictions.append(-100)
      print(f"Error occurred with earnings_hist: {e}")

  #if "earnings_hist_est" in successful:
  try:
      pred = clf_earnings_hist_est.predict_proba(earnings_hist_est)
      #weighted_prediction.append(pred[0,1]*(4225/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(4225/32906))
      predictions.append(-100)
      print(f"Error occurred with earnings_hist_est: {e}")

  #if "earnings_hist_est_eps" in successful:
  try:
      pred = clf_earnings_hist_est_eps.predict_proba(earnings_hist_est_eps)
      #weighted_prediction.append(pred[0,1]*(3844/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3844/32906))
      predictions.append(-100)
      print(f"Error occurred with earnings_hist_est_eps: {e}")

  #if "rev_est" in successful:
  try:
      pred = clf_rev_est.predict_proba(rev_est)
      #weighted_prediction.append(pred[0,1]*(2401/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(2401/32906))
      predictions.append(-100)
      print(f"Error occurred with rev_est: {e}")

  #if "meta" in successful:
  try:
      pred = clf_meta.predict_proba(meta)
      #weighted_prediction.append(pred[0,1]*(3364/32906))
      predictions.append(pred[0,1])
  except Exception as e:
      #weighted_prediction.append(0.5*(3364/32906))
      predictions.append(-100)
      print(f"Error occurred with meta: {e}")


predictions = pd.DataFrame({"":predictions}).transpose()
final_estimator_pred = clf_final_estimator.predict_proba(predictions)
if final_estimator_pred[0,1] >0.91:
  prediction = 1
  print("🔹", prediction,"🔹", "~83% accuracy 🔹")

elif final_estimator_pred[0,1]<0.1:
  prediction = -1
  print("🔹", prediction,"🔹", "~86% accuracy 🔹")

else:
    if final_estimator_pred[0,1] >0.8:
      prediction = 1
      print("🔹", prediction,"🔹", "~69% accuracy 🔹")

    elif final_estimator_pred[0,1]<0.15:
      prediction = -1
      print("🔹", prediction,"🔹", "~69% accuracy 🔹")
    else:
      prediction = "unknown"
      print("🔸 Uncertain 🔸")

In [ ]:
ticker = yf.Ticker("aapl")
ticker = ticker.earnings_estimate
print(ticker.shape, ticker)

(4, 6)             avg   low  high  yearAgoEps  numberOfAnalysts  growth
period                                                           
0q      1.75844  1.63  1.82      0.9700                29  0.8128
+1q     2.48513  2.29  2.67      2.4000                23  0.0355
0y      7.36760  7.09  7.47      6.0800                40  0.2118
+1y     7.98158  7.13  9.00      7.3676                40  0.0833


In [ ]:
df_earnings_est

,avg_1,low_1,high_1,numberOfAnalysts_1,avg_2,low_2,high_2,numberOfAnalysts_2,avg_3,low_3,high_3,numberOfAnalysts_3,avg_4,low_4,high_4,numberOfAnalysts_4
0,0.42604,-0.08,0.6777,18,0.59582,0.09,1.00345,18,3.28272,2.040,4.09855,19,4.67772,2.84000,7.33553,18
1,-0.25083,-0.40,-0.1400,12,-0.02905,-0.22,0.05000,12,-0.42807,-0.880,-0.24000,17,-0.19185,-0.84000,0.07000,16
2,1.29455,1.25,1.3200,33,1.20223,1.12,1.30000,31,5.09029,4.740,5.52000,43,5.62721,4.70000,6.22000,43
3,2.21126,2.01,2.3800,8,1.20013,0.92,1.41000,8,7.16837,6.300,9.38000,12,5.98424,4.05758,7.23000,14
4,0.92636,0.46,1.3000,10,1.51123,1.33,1.69000,10,3.87868,3.803,3.95000,18,4.36711,4.30000,4.64000,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,4.70333,4.14,4.9600,33,3.84581,3.74,4.10392,32,15.00808,14.900,15.38465,37,16.32581,15.66000,17.50000,37
540,0.13000,0.13,0.1300,1,0.15000,0.15,0.15000,1,0.65000,0.650,0.65000,1,0.60500,0.56000,0.65000,2
541,0.02000,0.02,0.0200,1,-0.03000,-0.03,-0.03000,1,1.25000,1.250,1.25000,1,1.29000,1.29000,1.29000,1
542,0.41592,0.38,0.4500,14,0.43831,0.40,0.48000,13,1.64035,1.550,1.75000,18,1.90951,1.71000,2.04000,18


In [ ]:
ticker = yf.Ticker('ko')
df = ticker.quarterly_cash_flow

In [ ]:
df

,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30
Free Cash Flow,4.564000e+09,3.369000e+09,-5.511000e+09,3.148000e+09,-1.728000e+09
Repurchase Of Capital Stock,-1.720000e+08,-1.020000e+08,-3.700000e+08,-5.670000e+08,-3.540000e+08
Repayment Of Debt,-1.536000e+09,-1.031000e+09,-1.599000e+09,-1.608000e+09,-3.191000e+09
Issuance Of Debt,-4.660000e+08,-1.160000e+08,5.436000e+09,7.630000e+08,4.466000e+09
Issuance Of Capital Stock,2.000000e+07,6.400000e+07,1.590000e+08,3.000000e+07,2.800000e+08
Capital Expenditure,-4.790000e+08,-4.420000e+08,-3.090000e+08,-8.030000e+08,-4.690000e+08
End Cash Position,1.336400e+10,1.020300e+10,8.417000e+09,1.148800e+10,1.416100e+10
Other Cash Adjustment Outside Changein Cash,NaN,NaN,-3.970000e+08,NaN,NaN
Beginning Cash Position,1.020300e+10,8.417000e+09,1.148800e+10,1.416100e+10,1.391300e+10
Effect Of Exchange Rate Changes,3.000000e+06,1.690000e+08,1.630000e+08,-3.570000e+08,9.100000e+07
